# ReviewLens — Data Audit & Preparation

Notebook tahap 01 memeriksa struktur, kualitas, label, dan duplikasi lima dataset,
melakukan normalisasi konservatif, lalu menyimpan dan membaca ulang hasil cleaning.

**Input:** lima file di `data/raw/` (hanya dibaca).
**Output:** lima `*_clean.csv` di `data/processed/` dan laporan audit di `outputs/reports/`.
Notebook ini belum melakukan AI training. Jalankan seluruh cell berurutan dengan **Restart Kernel and Run All**.


## 2. Project setup
Cari penanda proyek dari working directory dan parent-nya. Bootstrap hanya menambahkan
`src/` ke import path; konfigurasi utama tetap berada di `reviewlens/config.py`.
Dapat dijalankan dari root workspace/repository atau direktori notebook.


In [1]:
from pathlib import Path
import sys

project_root = None
for candidate in (Path.cwd(), *Path.cwd().parents):
    if (candidate / "src" / "reviewlens" / "config.py").is_file() and (candidate / "data" / "raw").is_dir():
        project_root = candidate.resolve()
        break
if project_root is None:
    raise FileNotFoundError("ReviewLens project root not found. Open this notebook from the project directory.")

source_dir = str(project_root / "src")
if source_dir not in sys.path:
    sys.path.insert(0, source_dir)

from reviewlens.config import PROJECT_ROOT, DATA_RAW_DIR, DATA_PROCESSED_DIR

print("PROJECT_ROOT:", PROJECT_ROOT)
print("RAW_DATA_DIR:", DATA_RAW_DIR)
print("PROCESSED_DATA_DIR:", DATA_PROCESSED_DIR)


PROJECT_ROOT: C:\Users\LENOVO\Documents\GitHub\ImpactHack2026-AI-Training
RAW_DATA_DIR: C:\Users\LENOVO\Documents\GitHub\ImpactHack2026-AI-Training\data\raw
PROCESSED_DATA_DIR: C:\Users\LENOVO\Documents\GitHub\ImpactHack2026-AI-Training\data\processed


## 3. Imports
Library tahap pertama: pandas, NumPy, library standar, serta helper data yang dapat
digunakan kembali dari aplikasi Python. Semua statistik dihitung dari dataset yang dibaca.


In [2]:
import json
import pandas as pd
import numpy as np
from IPython.display import display

from reviewlens.config import (
    DATASET_PATHS, PROCESSED_PATHS, TEXT_COLUMNS, LABEL_COLUMNS,
    LABEL_AUDIT_COLUMNS, DEDUPLICATION_KEYS, REPORT_DIR,
)
from reviewlens.data_utils import (
    validate_dataset_files, load_dataset, audit_dataframe, add_clean_text,
    duplicate_summary, find_label_conflicts, find_category_variants,
    label_distribution, clean_dataframe, cleaning_summary,
    save_csv, validate_saved_csv, file_sha256,
)

pd.set_option("display.max_colwidth", 100)
pd.set_option("display.max_columns", 12)
pd.set_option("display.max_rows", 25)
print(f"pandas {pd.__version__} | numpy {np.__version__}")


pandas 3.0.6 | numpy 2.5.3


## 4. File validation
Pastikan semua input ada sebelum loading. Hash SHA-256 dicatat agar notebook dapat
memastikan byte file raw tidak berubah selama pipeline. Nama sumber `challange.json` dipertahankan.


In [3]:
for path in DATASET_PATHS.values():
    print(f"[{'OK' if path.is_file() else 'MISSING'}] {path.name}")
validate_dataset_files(DATASET_PATHS)
raw_hashes_before = {name: file_sha256(path) for name, path in DATASET_PATHS.items()}


[OK] simple.json
[OK] challange.json
[OK] labeledReview.datasetFix.json
[OK] fake_reviews.csv
[OK] tokopedia-product-reviews-2019.csv


## 5. Load dataset
Ketiga JSON aktual berbentuk array of records. CSV dibaca dengan pandas. ID produk/toko
dan `sold` dibaca sebagai string agar leading zeros dan format seperti `3,2rb` tidak berubah.
Hanya field CSV kosong yang dianggap missing; teks literal seperti `NA` dipertahankan.


In [4]:
raw_datasets = {name: load_dataset(path) for name, path in DATASET_PATHS.items()}
for name, dataframe in raw_datasets.items():
    print(f"{name}: {dataframe.shape}")


simple: (17000, 3)
challenge: (4840, 4)
labeled: (11606, 3)
fake: (40432, 4)
tokopedia: (40607, 8)


## 6. Schema audit
Exact duplicates dihitung dari **seluruh kolom asli**, sebelum normalisasi. Jumlah
unique text mengabaikan nilai null. Tampilkan hanya tiga contoh per dataset.


In [5]:
audits = {}
schema_tables = []
for name, dataframe in raw_datasets.items():
    audit = audit_dataframe(dataframe, TEXT_COLUMNS[name])
    audits[name] = audit
    print(f"\nDataset: {name}")
    print("Shape:", (audit["rows"], audit["columns_count"]))
    print("Columns:", audit["columns"])
    schema = pd.DataFrame({"dtype": audit["dtypes"], "missing_values": audit["missing_values"]})
    schema.index.name = "column"
    display(schema)
    schema_tables.append(schema.reset_index().assign(dataset=name))
    print("Exact duplicates:", audit["exact_duplicates"])
    print("Unique text count:", audit["unique_text_count"])
    display(dataframe.head(3))

audit_summary = pd.DataFrame([
    {"dataset": name, "rows": audit["rows"], "columns": audit["columns_count"],
     "missing_cells": sum(audit["missing_values"].values()),
     "exact_duplicates": audit["exact_duplicates"], "unique_text_count": audit["unique_text_count"]}
    for name, audit in audits.items()
])
display(audit_summary)



Dataset: simple
Shape: (17000, 3)
Columns: ['comment', 'rating', 'sentiment']


,dtype,missing_values
column,,
comment,str,0
rating,int64,0
sentiment,str,0


Exact duplicates: 14040
Unique text count: 2193


,comment,rating,sentiment
0,"Wow, senang banget ditipu!",1,negative
1,Shock! Ga nyangka bakal sekeren ini produknya.,5,positive
2,"Top reviewnya, kecewa barangnya!",1,negative



Dataset: challenge
Shape: (4840, 4)
Columns: ['comment', 'sentiment', 'category', 'rating']


...


Dataset: tokopedia
Shape: (40607, 8)
Columns: ['text', 'rating', 'category', 'product_name', 'product_id', 'sold', 'shop_id', 'product_url']


,dtype,missing_values
column,,
text,string,0
rating,int64,0
category,string,0
product_name,string,0
product_id,string,0
sold,string,14
shop_id,string,0
product_url,string,0


Exact duplicates: 0
Unique text count: 37301


,text,rating,category,product_name,product_id,sold,shop_id,product_url
0,Barang sesuai pesanan dan cepat sampai,5,pertukangan,Staples Dekorasi Staples Kayu + Refill 8mm - Staples Gun 4-8 mm C Mart,418660637,1,1740837,https://www.tokopedia.com/shakaonline87/staples-dekorasi-staples-kayu-refill-8mm-staples-gun-4-8...
1,Barang bagus harga murah,5,pertukangan,STAPLE GUN ATS 3 WAY TACKER - STAPLES JOK TEMBAK HEKTER 022708,416032545,11,1477109,https://www.tokopedia.com/juraganperkakas/staple-gun-ats-3-way-tacker-staples-jok-tembak-hekter-...
2,Paket rapi...mantap....cepat....sampe ke tujuan,5,pertukangan,STAPLE GUN ATS 3 WAY TACKER - STAPLES JOK TEMBAK HEKTER 022708,416032545,11,1477109,https://www.tokopedia.com/juraganperkakas/staple-gun-ats-3-way-tacker-staples-jok-tembak-hekter-...


,dataset,rows,columns,missing_cells,exact_duplicates,unique_text_count
0,simple,17000,3,0,14040,2193
1,challenge,4840,4,0,0,4840
2,labeled,11606,3,0,4,11599
3,fake,40432,4,0,12,40412
4,tokopedia,40607,8,14,0,37301


## 7. Label distribution
Kolom opsional yang tidak tersedia dilaporkan dan dilewati pada bagian distribusi.
Cleaning tetap memerlukan kolom kuncinya dan memberi error jelas bila schema berubah.
`sentimen` dipertahankan sebagai label binary 0/1 tanpa mengarang pemetaan semantiknya.
`CG` berarti computer-generated, `OR` berarti original review; label tersebut tidak diubah.


In [6]:
distribution_tables = []
for name, dataframe in raw_datasets.items():
    columns = LABEL_AUDIT_COLUMNS[name]
    for column in columns:
        if column not in dataframe.columns:
            print(f"[SKIP] {name}.{column}: column not found")
    distribution = label_distribution(dataframe, columns)
    print(f"\n{name}")
    display(distribution)
    distribution_tables.append(distribution.assign(dataset=name, stage="raw"))



simple


,column,value,count
0,sentiment,positive,5667
1,sentiment,neutral,5667
2,sentiment,negative,5666



challenge


,column,value,count
0,sentiment,positive,1813
1,sentiment,negative,1804
2,sentiment,neutral,1223
3,category,short_vague,1783
4,category,aspect_based,1485
5,category,ambiguous,775
6,category,contextual,215
7,category,negation,97
8,category,typos_informal,93
9,category,mixed_sentiment,79



labeled


,column,value,count
0,sentimen,0,5803
1,sentimen,1,5803



fake


,column,value,count
0,label,CG,20216
1,label,OR,20216



tokopedia


,column,value,count
0,rating,5,30311
1,rating,4,7546
2,rating,3,1825
3,rating,1,543
4,rating,2,382
5,category,elektronik,15897
6,category,fashion,8910
7,category,olahraga,7838
8,category,handphone,6136
9,category,pertukangan,1826


## 8. Text normalization
Tambahkan `clean_text` pada salinan dataframe: handle null, string conversion, decode HTML
entity, lowercase, strip, dan rapikan whitespace. URL replacement tersedia di helper tetapi
**nonaktif** pada pipeline ini. Emoji, punctuation, negation, slang, dan semua kolom sumber dipertahankan.
Untuk labeled, gunakan `review` Bahasa Indonesia; `translate` tetap metadata.


In [7]:
normalized_datasets = {
    name: add_clean_text(dataframe, TEXT_COLUMNS[name])
    for name, dataframe in raw_datasets.items()
}
for name, dataframe in normalized_datasets.items():
    print(f"\n{name}")
    display(dataframe[[TEXT_COLUMNS[name], "clean_text"]].head(3))



simple


,comment,clean_text
0,"Wow, senang banget ditipu!","wow, senang banget ditipu!"
1,Shock! Ga nyangka bakal sekeren ini produknya.,shock! ga nyangka bakal sekeren ini produknya.
2,"Top reviewnya, kecewa barangnya!","top reviewnya, kecewa barangnya!"



challenge


,comment,clean_text
0,Wah mantap banget nih ditipu! Terima kasih seller 😊,wah mantap banget nih ditipu! terima kasih seller 😊
1,"Bagus sekali, barang rusak semua! Top markotop!","bagus sekali, barang rusak semua! top markotop!"
2,Pelayanan terbaik! Nungguin 2 minggu cuma buat barang ga dateng 🙏,pelayanan terbaik! nungguin 2 minggu cuma buat barang ga dateng 🙏



labeled


,review,clean_text
0,Suara dan mic puas. Hanya saya pesan yg nyala dan sudah sy pastikan lg klik yg nyala tp yg datan...,suara dan mic puas. hanya saya pesan yg nyala dan sudah sy pastikan lg klik yg nyala tp yg datan...
1,Good. Mudah-mudahan awet. Makasih,good. mudah-mudahan awet. makasih
2,"kondisi dus robek2, tolong di perhatikan lagi","kondisi dus robek2, tolong di perhatikan lagi"



fake


,text_,clean_text
0,"Love this! Well made, sturdy, and very comfortable. I love it!Very pretty","love this! well made, sturdy, and very comfortable. i love it!very pretty"
1,"love it, a great upgrade from the original. I've had mine for a couple of years","love it, a great upgrade from the original. i've had mine for a couple of years"
2,This pillow saved my back. I love the look and feel of this pillow.,this pillow saved my back. i love the look and feel of this pillow.



tokopedia


,text,clean_text
0,Barang sesuai pesanan dan cepat sampai,barang sesuai pesanan dan cepat sampai
1,Barang bagus harga murah,barang bagus harga murah
2,Paket rapi...mantap....cepat....sampe ke tujuan,paket rapi...mantap....cepat....sampe ke tujuan


## 9. Duplicate analysis setelah normalization
`duplicate_clean_text_count` adalah jumlah kemunculan tambahan setelah kemunculan pertama,
tanpa memperhitungkan label, category, atau produk. Angka ini **bukan** jumlah baris yang otomatis dihapus.
Teks dengan label bertentangan dipertahankan untuk pemeriksaan; tidak direlabel otomatis.
Periksa juga variasi category challenge dan overlap simple/challenge untuk perencanaan hard evaluation set.


In [8]:
duplicate_analysis = pd.DataFrame([
    {"dataset": name, **duplicate_summary(dataframe)}
    for name, dataframe in normalized_datasets.items()
])
display(duplicate_analysis)

label_conflicts = {}
for name, label_column in LABEL_COLUMNS.items():
    conflicts = find_label_conflicts(normalized_datasets[name], label_column)
    label_conflicts[name] = conflicts
    print(f"{name}: {conflicts['clean_text'].nunique()} texts with conflicting labels")
    if not conflicts.empty:
        display(conflicts.head(5))

category_variants = find_category_variants(normalized_datasets["challenge"], "sentiment")
print("Challenge text/sentiment groups with different categories:", len(category_variants))
display(category_variants.head(5))

overlap_texts = sorted(
    (set(normalized_datasets["simple"]["clean_text"]) & set(normalized_datasets["challenge"]["clean_text"])) - {""}
)
overlap_report = pd.DataFrame({"clean_text": overlap_texts})
print("Simple/challenge shared normalized texts:", len(overlap_report))
display(overlap_report.head(5))

tokopedia = normalized_datasets["tokopedia"]
print("Tokopedia repeated text within the same product:",
      int(tokopedia.duplicated(["product_id", "clean_text"]).sum()))
print("Tokopedia missing sold values:", int(tokopedia["sold"].isna().sum()))


,dataset,rows_total,unique_clean_text,duplicate_clean_text_count,empty_clean_text_count
0,simple,17000,2193,14807,0
1,challenge,4840,4750,90,0
2,labeled,11606,11588,18,0
3,fake,40432,40405,27,0
4,tokopedia,40607,36866,3741,0


simple: 0 texts with conflicting labels
challenge: 1 texts with conflicting labels


,comment,sentiment,category,rating,clean_text
127,Oke banget,positive,short_vague,5,oke banget
1120,oke banget,neutral,contextual,3,oke banget


labeled: 1 texts with conflicting labels


,review,sentimen,translate,clean_text
5645,barang bagus pengiriman agak lama,0,good item long delivery,barang bagus pengiriman agak lama
10771,Barang bagus\nPengiriman agak lama,1,good item. delivery took a while,barang bagus pengiriman agak lama


...

Challenge text/sentiment groups with different categories: 44


,clean_text,sentiment,categories,category_count
0,bagus sekali produknya,positive,"[""aspect_based"", ""short_vague""]",2
1,barang standar,neutral,"[""ambiguous"", ""short_vague""]",2
2,biasa banget,neutral,"[""ambiguous"", ""contextual""]",2
3,biasa lah,neutral,"[""ambiguous"", ""contextual""]",2
4,biasa saja banget,neutral,"[""contextual"", ""short_vague""]",2


Simple/challenge shared normalized texts: 11


,clean_text
0,biasa aja sih
1,gitu aja
2,lumayan
3,oke aja
4,oke deh


Tokopedia repeated text within the same product: 53
Tokopedia missing sold values: 14


## 10. Cleaning
Kunci deduplication dipilih eksplisit:

| Dataset | Kunci / kebijakan |
| --- | --- |
| simple | `clean_text + sentiment` |
| challenge | `clean_text + sentiment + category` |
| labeled | `clean_text + sentimen` |
| fake | `clean_text + label` |
| tokopedia | Pertahankan seluruh baris, termasuk duplicate text |

**Penyesuaian challenge sesuai pilihan pengguna setelah audit:** category menjadi bagian kunci
karena ada teks dan sentimen sama dengan category berbeda. Tidak mengubah nilai category.
Untuk setiap kunci, simpan kemunculan pertama dalam urutan file; rating tidak menjadi filter.
Metadata non-kunci mengikuti baris pertama. Semua versi asli tetap tersedia di raw.
Jika ada text kosong, barisnya dipertahankan untuk pemeriksaan dan tidak digabung sebagai duplicate.


In [9]:
cleaned_datasets = {}
for name, dataframe in normalized_datasets.items():
    keys = DEDUPLICATION_KEYS[name]
    cleaned_datasets[name] = clean_dataframe(dataframe, keys)
    print(f"{name}: {'keep all rows' if keys is None else ' + '.join(keys)}")

# Tokopedia retains every original observation and all metadata.
pd.testing.assert_frame_equal(
    cleaned_datasets["tokopedia"], normalized_datasets["tokopedia"].reset_index(drop=True)
)
# Every observed challenge category per text/sentiment remains represented.
challenge_keys = list(DEDUPLICATION_KEYS["challenge"])
pd.testing.assert_frame_equal(
    normalized_datasets["challenge"][challenge_keys].drop_duplicates().reset_index(drop=True),
    cleaned_datasets["challenge"][challenge_keys].drop_duplicates().reset_index(drop=True),
)


simple: clean_text + sentiment
challenge: clean_text + sentiment + category
labeled: clean_text + sentimen
fake: clean_text + label
tokopedia: keep all rows


## 11. Cleaning summary
Jumlah sebelum/sesudah dan persentase removal dihitung langsung dari dataframe.
Distribusi setelah cleaning juga disimpan agar perubahan proporsi label dapat diperiksa.


In [10]:
summary = cleaning_summary(raw_datasets, cleaned_datasets)
display(summary)
for name, dataframe in cleaned_datasets.items():
    distribution = label_distribution(dataframe, LABEL_AUDIT_COLUMNS[name])
    distribution_tables.append(distribution.assign(dataset=name, stage="cleaned"))
label_distribution_report = pd.concat(distribution_tables, ignore_index=True)


,dataset,rows_before,rows_after,removed,removal_percent
0,simple,17000,2193,14807,87.10
1,challenge,4840,4795,45,0.93
2,labeled,11606,11589,17,0.15
3,fake,40432,40405,27,0.07
4,tokopedia,40607,40607,0,0.00


## 12. Save processed CSV dan laporan
Simpan `index=False`, `encoding="utf-8-sig"`. Menjalankan ulang notebook memperbarui hasil
processed dan laporan dengan kebijakan yang sama. Pipeline tidak menulis ke `data/raw/`.
CSV tidak membawa schema dtype: gunakan helper `load_dataset()` jika ingin mempertahankan ID sebagai string.


In [11]:
for name, dataframe in cleaned_datasets.items():
    path = save_csv(dataframe, PROCESSED_PATHS[name])
    print("Saved:", path.relative_to(PROJECT_ROOT))

reports = {
    "audit_summary.csv": audit_summary,
    "schema_audit.csv": pd.concat(schema_tables, ignore_index=True),
    "duplicate_summary.csv": duplicate_analysis,
    "label_distribution.csv": label_distribution_report,
    "cleaning_summary.csv": summary,
    "challenge_category_variants.csv": category_variants,
    "simple_challenge_overlap.csv": overlap_report,
}
reports.update({f"{name}_label_conflicts.csv": conflicts for name, conflicts in label_conflicts.items()})
for filename, report in reports.items():
    save_csv(report, REPORT_DIR / filename)

environment = {"python": sys.version.split()[0], "pandas": pd.__version__, "numpy": np.__version__}
(REPORT_DIR / "environment.json").write_text(json.dumps(environment, indent=2) + "\n", encoding="utf-8")


Saved: data\processed\simple_clean.csv
Saved: data\processed\challenge_clean.csv
Saved: data\processed\labeled_clean.csv


Saved: data\processed\fake_clean.csv


Saved: data\processed\tokopedia_clean.csv


66

## 13. Validate saved files
Baca ulang seluruh CSV. Pastikan file ada, dataset tidak kosong, row count dan urutan kolom
sesuai, `clean_text` tersedia, serta **semua nilai serialized** cocok dengan dataframe hasil cleaning.
Readback string mempertahankan teks literal `NA`, Unicode, leading zeros, dan field kosong.
Field kosong CSV tidak membedakan empty string dari null; perbedaannya tersedia di audit/raw.
Terakhir, verifikasi raw menggunakan SHA-256 sebelum dan sesudah eksekusi.


In [12]:
validated_datasets = {}
for name, dataframe in cleaned_datasets.items():
    saved = validate_saved_csv(PROCESSED_PATHS[name], dataframe)
    validated_datasets[name] = saved
    empty_count = int(saved["clean_text"].eq("").sum())
    print(f"[OK] {PROCESSED_PATHS[name].name} — {len(saved):,} rows; empty clean_text: {empty_count}")

integrity_rows = []
for name, path in DATASET_PATHS.items():
    after_hash = file_sha256(path)
    if raw_hashes_before[name] != after_hash:
        raise AssertionError(f"Raw dataset changed during preparation: {path.name}")
    integrity_rows.append({"dataset": name, "file": str(path.relative_to(PROJECT_ROOT)),
                           "bytes": path.stat().st_size, "sha256": after_hash, "unchanged": True})
save_csv(pd.DataFrame(integrity_rows), REPORT_DIR / "raw_integrity.csv")
print("[OK] All raw dataset hashes unchanged.")


[OK] simple_clean.csv — 2,193 rows; empty clean_text: 0
[OK] challenge_clean.csv — 4,795 rows; empty clean_text: 0


[OK] labeled_clean.csv — 11,589 rows; empty clean_text: 0


[OK] fake_clean.csv — 40,405 rows; empty clean_text: 0


[OK] tokopedia_clean.csv — 40,607 rows; empty clean_text: 0
[OK] All raw dataset hashes unchanged.


## 14. Final summary
Hasil berikut tampil setelah seluruh validasi lulus. Konflik label dan overlap antar dataset
tetap perlu diperiksa sebelum menetapkan split train/evaluation. Notebook 02 masih placeholder
dan baru dikerjakan setelah instruksi lanjutan.


In [13]:
print("Data preparation completed successfully.")
print("\nProcessed datasets:")
for path in PROCESSED_PATHS.values():
    print("-", path.relative_to(PROJECT_ROOT))
display(summary)
print("\nAudit reports:", REPORT_DIR.relative_to(PROJECT_ROOT))
print("Next step: 02_sentiment_model.ipynb (belum diimplementasikan / dijalankan).")


Data preparation completed successfully.

Processed datasets:
- data\processed\simple_clean.csv
- data\processed\challenge_clean.csv
- data\processed\labeled_clean.csv
- data\processed\fake_clean.csv
- data\processed\tokopedia_clean.csv


,dataset,rows_before,rows_after,removed,removal_percent
0,simple,17000,2193,14807,87.10
1,challenge,4840,4795,45,0.93
2,labeled,11606,11589,17,0.15
3,fake,40432,40405,27,0.07
4,tokopedia,40607,40607,0,0.00



Audit reports: outputs\reports
Next step: 02_sentiment_model.ipynb (belum diimplementasikan / dijalankan).
